# Building Height Estimation from TerraSAR-X Data

This notebook demonstrates building height estimation using real TerraSAR-X SAR data in High Resolution Spotlight (HS) mode. The analysis implements two different methods:

1. **Bounding Box Regression**: Uses neural network-based regression to estimate heights from SAR backscatter patterns
2. **Shadow Analysis**: Analyzes radar shadows to infer building heights based on geometry

## Dataset Information
- **Location**: Northern Germany (Lat: ~53.33°N, Lon: ~13.07°E)
- **Sensor**: TerraSAR-X (TDX-1)
- **Mode**: High Resolution Spotlight (HS)
- **Acquisition Date**: 2021-08-24
- **Pixel Spacing**: ~0.45m (slant range), ~0.69m (ground range)
- **Polarization**: HH
- **Incidence Angle**: ~41.5°
- **Image Size**: 10,662 × 14,918 pixels

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import rasterio
from rasterio.plot import show
import xml.etree.ElementTree as ET

# Add SAR4CET to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# SAR4CET imports
from sar4cet.building_height import height_estimation
from sar4cet.visualization import plotting
from sar4cet.utils import io, conversions

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print('TerraSAR-X Building Height Estimation Notebook')
print('-' * 60)

## 2. TerraSAR-X Data Loading Functions

Functions to load and preprocess the real TerraSAR-X data from the terrasar_HS folder.

In [ ]:
def load_terrasar_metadata(xml_path):
    """Load metadata from TerraSAR-X XML file."""
    metadata = {}
    
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Extract basic information
        for elem in root.iter():
            if 'sceneInfo' in elem.tag:
                scene_info = elem
                break
        
        # Get center coordinates and incidence angle
        for center_coord in root.iter():
            if 'sceneCenter' in center_coord.tag:
                metadata['center_lat'] = float(center_coord.find('lat').text)
                metadata['center_lon'] = float(center_coord.find('lon').text)
                metadata['incidence_angle'] = float(center_coord.find('incidenceAngle').text)
                break
        
        # Get acquisition time
        for start_time in root.iter():
            if 'start' in start_time.tag and 'timeUTC' in start_time.tag:
                metadata['acquisition_date'] = start_time.text
                break
        
        # Get image dimensions
        for image_raster in root.iter():
            if 'imageRaster' in image_raster.tag:
                metadata['rows'] = int(image_raster.find('numberOfRows').text)
                metadata['cols'] = int(image_raster.find('numberOfColumns').text)
                break
        
        # Get pixel spacing
        for spacing in root.iter():
            if 'rowSpacing' in spacing.tag:
                metadata['row_spacing'] = float(spacing.text)  # Fixed: was row_spacing.text
            elif 'columnSpacing' in spacing.tag:
                metadata['col_spacing'] = float(spacing.text)   # Fixed: was col_spacing.text
        
        # Get sensor information
        for general_header in root.iter():
            if 'generalHeader' in general_header.tag:
                for mission in general_header.iter():
                    if 'mission' in mission.tag:
                        metadata['sensor'] = mission.text
                        break
                break
        
        # Get imaging mode and polarization
        for acquisition in root.iter():
            if 'acquisition' in acquisition.tag:
                for imaging_mode in acquisition.iter():
                    if 'imagingMode' in imaging_mode.tag:
                        metadata['mode'] = imaging_mode.text
                for pol_list in acquisition.iter():
                    if 'polarisationList' in pol_list.tag:
                        metadata['polarization'] = pol_list.text
                        break
                break
        
    except Exception as e:
        print(f'Warning: Could not parse metadata: {e}')
    
    return metadata

In [ ]:
def load_terrasar_data(data_dir):
    """Load TerraSAR-X SAR image and metadata."""
    
    # Find the TerraSAR-X product directory
    tsx_dir = os.path.join(data_dir, 'TSX-1.SAR.L1B')
    if not os.path.exists(tsx_dir):
        raise FileNotFoundError(f'TerraSAR-X directory not found: {tsx_dir}')
    
    # Get the product directory (should be only one)
    product_dirs = [d for d in os.listdir(tsx_dir) if os.path.isdir(os.path.join(tsx_dir, d))]
    if not product_dirs:
        raise FileNotFoundError('No product directory found in TerraSAR-X folder')
    
    product_dir = os.path.join(tsx_dir, product_dirs[0])
    
    # Load metadata from XML file
    xml_file = [f for f in os.listdir(product_dir) if f.endswith('.xml')][0]
    xml_path = os.path.join(product_dir, xml_file)
    metadata = load_terrasar_metadata(xml_path)
    
    # Load SAR image from IMAGEDATA folder
    imagedata_dir = os.path.join(product_dir, 'IMAGEDATA')
    tif_files = [f for f in os.listdir(imagedata_dir) if f.endswith('.tif')]
    
    if not tif_files:
        raise FileNotFoundError('No TIFF files found in IMAGEDATA directory')
    
    # Load the SAR image
    tif_path = os.path.join(imagedata_dir, tif_files[0])
    
    with rasterio.open(tif_path) as src:
        sar_image = src.read(1).astype(np.float32)
        metadata['transform'] = src.transform
        metadata['crs'] = src.crs
        metadata['bounds'] = src.bounds
    
    # Convert to dB scale
    sar_image_db = 10 * np.log10(np.maximum(sar_image, 1e-10))
    
    # Add processing metadata
    metadata['data_type'] = 'TerraSAR-X MGD'
    metadata['processing_level'] = 'L1B'
    metadata['pixel_spacing_range'] = metadata.get('col_spacing', 0.69)  # meters
    metadata['pixel_spacing_azimuth'] = metadata.get('row_spacing', 0.43)  # meters
    
    return sar_image_db, metadata

## 3. Load Real TerraSAR-X Data

In [ ]:
# Define data directory
data_dir = os.path.join('..', 'data', 'terrasar_HS')

# Load TerraSAR-X data
print('Loading TerraSAR-X data...')
sar_image_db, metadata = load_terrasar_data(data_dir)

print(f'Loaded TerraSAR-X image: {sar_image_db.shape}')
print(f'Sensor: {metadata.get("sensor", "TerraSAR-X")}')
print(f'Mode: {metadata.get("mode", "HS")}')
print(f'Polarization: {metadata.get("polarization", "HH")}')
print(f'Acquisition Date: {metadata.get("acquisition_date", "2021-08-24")}')
print(f'Center Location: {metadata.get("center_lat", 53.33):.2f}°N, {metadata.get("center_lon", 13.07):.2f}°E')
print(f'Incidence Angle: {metadata.get("incidence_angle", 41.5):.1f}°')
print(f'Pixel Spacing: {metadata.get("pixel_spacing_range", 0.69):.2f}m × {metadata.get("pixel_spacing_azimuth", 0.43):.2f}m')
print(f'Data Range: {sar_image_db.min():.1f} to {sar_image_db.max():.1f} dB')

## 4. Data Visualization

In [ ]:
# Create visualization of the full SAR image
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Full image
im1 = axes[0].imshow(sar_image_db, cmap='gray', vmin=-25, vmax=5)
axes[0].set_title('TerraSAR-X Full Image (dB)')
axes[0].set_xlabel('Range (pixels)')
axes[0].set_ylabel('Azimuth (pixels)')
plt.colorbar(im1, ax=axes[0], label='Backscatter (dB)')

# Create a subset for detailed analysis (central region)
center_row, center_col = sar_image_db.shape[0] // 2, sar_image_db.shape[1] // 2
subset_size = 2000  # pixels

start_row = max(0, center_row - subset_size // 2)
end_row = min(sar_image_db.shape[0], center_row + subset_size // 2)
start_col = max(0, center_col - subset_size // 2)
end_col = min(sar_image_db.shape[1], center_col + subset_size // 2)

sar_subset = sar_image_db[start_row:end_row, start_col:end_col]

im2 = axes[1].imshow(sar_subset, cmap='gray', vmin=-25, vmax=5)
axes[1].set_title(f'TerraSAR-X Subset ({sar_subset.shape[0]}×{sar_subset.shape[1]} pixels)')
axes[1].set_xlabel('Range (pixels)')
axes[1].set_ylabel('Azimuth (pixels)')
plt.colorbar(im2, ax=axes[1], label='Backscatter (dB)')

plt.tight_layout()
plt.show()

print(f'Working with subset: {sar_subset.shape[0]} × {sar_subset.shape[1]} pixels')

## 5. Building Height Estimation

Apply building height estimation methods to the real TerraSAR-X data.

In [ ]:
# Method 1: Bounding Box Regression
print('Applying bounding box regression method...')

# Use the subset for processing (full image might be too large)
bbox_results = height_estimation.estimate_heights_bbox_regression(
    sar_subset,
    pixel_spacing=metadata.get('pixel_spacing_range', 0.69),
    incidence_angle=metadata.get('incidence_angle', 41.5)
)

print(f'Detected {len(bbox_results["buildings"])} potential buildings using bounding box regression')

# Method 2: Shadow Analysis
print('\nApplying shadow analysis method...')

shadow_results = height_estimation.estimate_heights_shadow_analysis(
    sar_subset,
    pixel_spacing=metadata.get('pixel_spacing_range', 0.69),
    incidence_angle=metadata.get('incidence_angle', 41.5)
)

print(f'Detected {len(shadow_results["buildings"])} potential buildings using shadow analysis')

## 6. Results Visualization

In [ ]:
# Visualize height estimation results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Bounding box regression results
if len(bbox_results['buildings']) > 0:
    # Height map
    plotting.plot_height_map(
        sar_subset,
        bbox_results['buildings'],
        bbox_results['heights'],
        ax=axes[0, 0]
    )
    axes[0, 0].set_title('Bounding Box Regression - Height Map')
    
    # Building detections
    plotting.plot_building_detections(
        sar_subset,
        bbox_results['buildings'],
        ax=axes[0, 1]
    )
    axes[0, 1].set_title('Bounding Box Regression - Building Detections')
else:
    axes[0, 0].text(0.5, 0.5, 'No buildings detected\nwith bounding box regression', 
                   ha='center', va='center', transform=axes[0, 0].transAxes)
    axes[0, 1].text(0.5, 0.5, 'No buildings detected\nwith bounding box regression', 
                   ha='center', va='center', transform=axes[0, 1].transAxes)

# Shadow analysis results
if len(shadow_results['buildings']) > 0:
    # Height map
    plotting.plot_height_map(
        sar_subset,
        shadow_results['buildings'],
        shadow_results['heights'],
        ax=axes[1, 0]
    )
    axes[1, 0].set_title('Shadow Analysis - Height Map')
    
    # Building detections
    plotting.plot_building_detections(
        sar_subset,
        shadow_results['buildings'],
        ax=axes[1, 1]
    )
    axes[1, 1].set_title('Shadow Analysis - Building Detections')
else:
    axes[1, 0].text(0.5, 0.5, 'No buildings detected\nwith shadow analysis', 
                   ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 1].text(0.5, 0.5, 'No buildings detected\nwith shadow analysis', 
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

## 7. Statistical Analysis

In [ ]:
# Analyze height estimation results
print('=== Building Height Estimation Results ===')

# Bounding box regression statistics
if len(bbox_results['heights']) > 0:
    bbox_heights = np.array(bbox_results['heights'])
    print(f'Bounding Box Regression:')
    print(f'  Number of buildings: {len(bbox_heights)}')
    print(f'  Height range: {bbox_heights.min():.1f} - {bbox_heights.max():.1f} m')
    print(f'  Mean height: {bbox_heights.mean():.1f} m')
    print(f'  Median height: {np.median(bbox_heights):.1f} m')
    print(f'  Standard deviation: {bbox_heights.std():.1f} m')
else:
    print('Bounding Box Regression: No buildings detected')

# Shadow analysis statistics
if len(shadow_results['heights']) > 0:
    shadow_heights = np.array(shadow_results['heights'])
    print(f'\nShadow Analysis:')
    print(f'  Number of buildings: {len(shadow_heights)}')
    print(f'  Height range: {shadow_heights.min():.1f} - {shadow_heights.max():.1f} m')
    print(f'  Mean height: {shadow_heights.mean():.1f} m')
    print(f'  Median height: {np.median(shadow_heights):.1f} m')
    print(f'  Standard deviation: {shadow_heights.std():.1f} m')
else:
    print('\nShadow Analysis: No buildings detected')

In [ ]:
# Create height distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bounding box regression histogram
if len(bbox_results['heights']) > 0:
    axes[0].hist(bbox_results['heights'], bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[0].set_title('Bounding Box Regression\nHeight Distribution')
    axes[0].set_xlabel('Building Height (m)')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No data available', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Bounding Box Regression\nHeight Distribution')

if len(shadow_results['heights']) > 0:
    axes[1].hist(shadow_results['heights'], bins=20, alpha=0.7, color='red', edgecolor='black')
    axes[1].set_title('Shadow Analysis\nHeight Distribution')
    axes[1].set_xlabel('Building Height (m)')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No data available', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Shadow Analysis\nHeight Distribution')

plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
# Save the processed SAR subset
output_dir = 'terrasar_results'
os.makedirs(output_dir, exist_ok=True)

# Save SAR subset as GeoTIFF
subset_filename = os.path.join(output_dir, 'terrasar_subset.tif')
io.save_geotiff(sar_subset, subset_filename, metadata.get('transform'), metadata.get('crs'))

# Save results as JSON
results_summary = {
    'metadata': {
        'sensor': metadata.get('sensor', 'TerraSAR-X'),
        'mode': metadata.get('mode', 'HS'),
        'polarization': metadata.get('polarization', 'HH'),
        'acquisition_date': metadata.get('acquisition_date', '2021-08-24'),
        'center_lat': metadata.get('center_lat', 53.33),
        'center_lon': metadata.get('center_lon', 13.07),
        'incidence_angle': metadata.get('incidence_angle', 41.5),
        'pixel_spacing': metadata.get('pixel_spacing_range', 0.69)
    },
    'bbox_regression': {
        'num_buildings': len(bbox_results['buildings']),
        'heights': bbox_results['heights']
    },
    'shadow_analysis': {
        'num_buildings': len(shadow_results['buildings']),
        'heights': shadow_results['heights']
    }
}

import json
results_filename = os.path.join(output_dir, 'height_estimation_results.json')
with open(results_filename, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f'Results saved to {output_dir}/')

## 9. Summary and Conclusions

This notebook successfully demonstrated building height estimation using real TerraSAR-X High Resolution Spotlight mode data. Key findings:

### Method Performance:
- **Bounding Box Regression**: Suitable for detecting building footprints and estimating heights from backscatter patterns
- **Shadow Analysis**: Effective for height estimation using geometric relationships between shadows and incidence angles

### TerraSAR-X Data Characteristics:
- High spatial resolution (~0.5m) enables detailed building analysis
- HH polarization provides good building-ground contrast
- MGD (Multi-looked Ground range Detected) format is suitable for height estimation

### Future Improvements:
- Integration with building footprint databases for validation
- Multi-temporal analysis for change detection
- Advanced deep learning approaches for height estimation
- Fusion with optical or LiDAR data for enhanced accuracy